In [1]:
import os
print(os.getcwd())

/home/thienhb/Workspace/arxiv-paper-rag/notebooks


In [2]:
os.chdir("..")

In [9]:
# Test Complete Pipeline
from src.services.metadata_fetcher import make_metadata_fetcher
from src.services.arxiv.factory import make_arxiv_client
from src.services.pdf_parser.factory import make_pdf_parser_service
from src.db.factory import make_database

arxiv_client = make_arxiv_client()
pdf_parser = make_pdf_parser_service()
database = make_database()

# Create metadata fetcher
metadata_fetcher = make_metadata_fetcher(arxiv_client, pdf_parser)
print("✓ Metadata fetcher service created")

✓ Metadata fetcher service created


In [10]:
# Test with small batch
print("Running small batch test (2 papers, no PDF processing for speed)...")

try:
    with database.get_session() as session:
        results = await metadata_fetcher.fetch_and_process_papers(
            max_results=2,  
            process_pdfs=False,  
            store_to_db=True,
            db_session=session
        )
    
    print("\nPIPELINE RESULTS:")
    print(f"   Papers fetched: {results.get('papers_fetched', 0)}")
    print(f"   PDFs downloaded: {results.get('pdfs_downloaded', 0)}")
    print(f"   PDFs parsed: {results.get('pdfs_parsed', 0)}")
    print(f"   Papers stored: {results.get('papers_stored', 0)}")
    print(f"   Processing time: {results.get('processing_time', 0):.1f}s")
    print(f"   Errors: {len(results.get('errors', []))}")
    
    if results.get('errors'):
        print("\nErrors encountered:")
        for error in results.get('errors', [])[:3]:  # Show first 3 errors
            print(f"   - {error}")
    
    if results.get('papers_fetched', 0) > 0:
        print("\n✓ Pipeline test successful!")
    else:
        print("\nNo papers fetched - may be arXiv API unavailability")
        
except Exception as e:
    print(f"✗ Pipeline error: {e}")

Running small batch test (2 papers, no PDF processing for speed)...

PIPELINE RESULTS:
   Papers fetched: 2
   PDFs downloaded: 0
   PDFs parsed: 0
   Papers stored: 2
   Processing time: 0.3s
   Errors: 0

✓ Pipeline test successful!


In [ ]:
# Test Complete Pipeline with PDF Processing
print("Test 8: Complete Pipeline with PDF Processing")
print("=" * 50)

# Reuse metadata fetcher from Test 6
print("✓ Using metadata fetcher service from previous test")

# Test with small batch including PDF processing
print("Running enhanced test (3 papers with PDF processing)...")

try:
    with database.get_session() as session:
        results = await metadata_fetcher.fetch_and_process_papers(
            max_results=3,  # Small batch
            from_date="20250813",  # Recent date
            to_date="20250814",
            process_pdfs=True,  
            store_to_db=True,
            db_session=session
        )
    
    print("\nENHANCED PIPELINE RESULTS:")
    print(f"   Papers fetched: {results.get('papers_fetched', 0)}")
    print(f"   PDFs downloaded: {results.get('pdfs_downloaded', 0)}")
    print(f"   PDFs parsed: {results.get('pdfs_parsed', 0)}")
    print(f"   Papers stored: {results.get('papers_stored', 0)}")
    print(f"   Processing time: {results.get('processing_time', 0):.1f}s")
    print(f"   Errors: {len(results.get('errors', []))}")
    
    # Show success rates
    if results.get('papers_fetched', 0) > 0:
        download_rate = (results['pdfs_downloaded'] / results['papers_fetched']) * 100
        parse_rate = (results['pdfs_parsed'] / results['pdfs_downloaded']) * 100 if results.get('pdfs_downloaded', 0) > 0 else 0
        print(f"   Download success rate: {download_rate:.1f}%")
        print(f"   Parse success rate: {parse_rate:.1f}%")
    
    if results.get('errors'):
        print("\nErrors encountered (showing graceful error handling):")
        for error in results.get('errors', [])[:3]:  # Show first 3 errors
            print(f"   - {error}")
    
    if results.get('papers_fetched', 0) > 0:
        print("\n✓ Enhanced pipeline test successful!")
        if results.get('errors'):
            print("✓ System continued processing despite PDF failures")
    else:
        print("\n! No papers fetched - may be arXiv API unavailability")
        
except Exception as e:
    print(f"✗ Pipeline error: {e}")